In [13]:
import json
from openai import OpenAI

from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents

openai_client=OpenAI()
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

def search(query):
    return index.search(query=query, num_results=5)

def llm_structured(user_prompt, instructions, output_format):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_format
    )

    return response.output_parsed

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")



Indexed 382 chunks from 95 documents


In [14]:
from typing import Literal
from pydantic import  BaseModel, Field 

class RAGResponse(BaseModel):
    answer: str =Field(description="The main answer to the user s question")
    found_answer: bool = Field(description="true if relevant information was found")
    confidence: float = Field(description="confidence score from 0.0 to 1.0")
    answer_type: Literal["how to", "explanation", "troubleshooting", "comparison", "reference"]=Field(description="Category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [15]:
def rag(query, output_format=RAGResponse):
    search_results=search(query)
    prompt= build_prompt(query,search_results)
    return llm_structured(prompt, instructions, output_format)



In [16]:
query = 'llm as a judge'

answer = rag(query)

print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer_type: {answer.answer_type}")
print(f"answer: {answer.answer[:100]}...")

found_answer: True
confidence: 0.9
answer_type: explanation
answer: In this tutorial, an LLM (Large Language Model) is used as a judge to evaluate text based on custom ...


In [18]:
from sentence_transformers import  SentenceTransformer
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 26281.38it/s]


In [20]:
q1 = 'install Evidently locally' # -
q2 = 'how can I set up evidently in my projects?' # -> [1000+]

In [21]:
v1=model.encode(q1)
v1[:20]


array([ 0.07184872, -0.02879701,  0.06123491, -0.0181076 ,  0.12533103,
       -0.04134634, -0.06354982, -0.08986842, -0.04142817, -0.04465349,
        0.06421643,  0.02077088,  0.00219168,  0.00508835, -0.00833957,
       -0.03770019,  0.03059424, -0.04844618,  0.06848746, -0.01030121],
      dtype=float32)

In [22]:
v1.shape


(384,)

In [23]:
v2=model.encode(q2)
v1.dot(v2)


np.float32(0.43477964)

In [24]:
q3="how do i bake pretzels"
v3=model.encode(q3)
v1.dot(v3)

np.float32(0.021079447)

In [25]:
texts = []

for doc in chunked_docs:
    title = doc.get('title', '')
    description = doc.get('description', '')
    content = doc.get('content', '')

    text = title + " " + description + " " + content
    texts.append(text.strip())

In [26]:
embeddings=model.encode(texts, show_progress_bar=True)

Batches: 100%|██████████| 12/12 [00:07<00:00,  1.61it/s]


In [27]:
embeddings.shape 

(382, 384)

In [28]:
embeddings


array([[-0.04849767,  0.06601194,  0.01094062, ...,  0.03090533,
         0.06879071,  0.05070842],
       [-0.03232048, -0.01947774, -0.01677319, ...,  0.01519909,
         0.02379616, -0.00774651],
       [-0.02108488, -0.05657952,  0.00483413, ..., -0.04806894,
         0.07104321,  0.04317405],
       ...,
       [-0.00160591, -0.04139433, -0.05660459, ...,  0.05727461,
         0.09037887, -0.02732164],
       [-0.02243879, -0.00113356, -0.05355773, ...,  0.04080833,
         0.08891265, -0.00143775],
       [-0.06048304, -0.05174245,  0.01422593, ...,  0.00431385,
         0.09889203,  0.0039377 ]], shape=(382, 384), dtype=float32)

In [29]:
scores=embeddings.dot(v1)
scores [:15]


array([ 0.07959686,  0.12935543,  0.18665004,  0.26054296,  0.08233509,
       -0.02807789, -0.07214389, -0.05210186, -0.01589219,  0.00971279,
       -0.05274082, -0.10643847,  0.04062283,  0.06344493, -0.04217603],
      dtype=float32)

In [30]:
from minsearch import VectorSearch
vindex=VectorSearch(keyword_fields=['filename'])
vindex.fit(embeddings,chunked_docs)


In [31]:
def vector_search(query,num_results=5) :
    vector=model.encode(query)
    return vindex.search(vector,num_results=num_results)

def rag_vector(query, output_format=RAGResponse):
    search_results=vector_search(query)
    prompt=build_prompt(query, search_results)
    return llm_structured(prompt, instructions,output_format=RAGResponse)






In [33]:
answer=rag_vector(query)
print(f"Query: {query}")
print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer: {answer.answer[:200]}...")

Query: llm as a judge
found_answer: True
confidence: 0.9
answer: The context describes how to use a Large Language Model (LLM) as a judge to evaluate text against custom criteria. There are two main approaches: **Reference-based**, where responses are compared to a...


In [34]:
def hybrid_search(query):
    vector_results = vector_search(query)
    text_results = search(query)
    return vector_results + text_results
results = hybrid_search(query)
len(results)

10

In [35]:
results

[{'start': 0,
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different cont